# Home Safety Risk Ranking — Simplified Learner Recipe
## ADAPTED FOR STATION GROUND   LEVEL

This notebook guides you through building a **risk-ranking model** for the home-safety dataset.

The goal is **not** to create a perfect yes/no classifier. The goal is to sort records from highest risk to lowest risk so that the top records contain more true incidents than we would get by random selection.

This is important because the target is imbalanced. When positives are rare, accuracy and a default 0.5 threshold can be misleading. Here we care about questions like:

- If we inspect the **top 50** highest-risk records, how many true incidents do we capture?
- Is the top 50 list better than random selection?
- Would top 100 or top 200 be more realistic operationally?

This simplified version focuses on **one model family: Random Forest**.

## Before you start

You only need to make a small number of decisions:

1. Update the data file path.
2. Confirm the target column name.
3. Check the target is correctly encoded as 0/1.
4. Review the list of columns removed from modelling to avoid leakage.
5. Run the RandomizedSearchCV section and interpret the top-K results.

Avoid changing too many things at once. Get the notebook running first, then improve it.

## 1. Imports

Run this cell first. It loads the packages needed for the workflow.

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from scipy.stats import randint

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Simple configuration

Update only the values in this cell.

The most important settings are:

- `DATA_PATH`: where the dataset is saved.
- `TARGET_RAW`: the original target column in the dataset.
- `TOP_K_VALUES`: the ranked list sizes to evaluate.

In [2]:
# TODO 1: update this path to match where your dataset is saved.
#DATA_PATH = Path("df_new.xlsx")
#DATA_PATH = Path("C:/Users/pw347789/OneDrive - Oxfordshire County Council/Desktop/Cambridge Spark/PROJECT/HomeSafety_2/RawData/PYTHON_OUTPUTS/df_new.xlsx")
#DATA_PATH = Path("C:/Users/pw347789/OneDrive - Oxfordshire County Council/Desktop/Cambridge Spark/PROJECT/HomeSafety_2/RawData/Household_data/RESIDENTIAL_OXON_anon.xlsx")
DATA_PATH = Path("C:/Users/pw347789/OneDrive - Oxfordshire County Council/Desktop/Cambridge Spark/PROJECT/HomeSafety_2/RawData/PYTHON_OUTPUTS/OXON_HH_DATA.xlsx")

# TODO 2: confirm this is the correct target column in your dataset.
TARGET_RAW = "Incident?"


# This will be the cleaned 0/1 target column used for modelling.
TARGET = "target_incident"

# These are the ranked list sizes we want to evaluate.
# Keep this simple at first.
TOP_K_VALUES = [50, 100, 200, 500]

# Random seed so results are reproducible.
RANDOM_STATE = 42

# Data split sizes.
TEST_SIZE = 0.20
VALIDATION_SIZE_WITHIN_TRAIN = 0.25

# Random search size.
# Start with 25. Increase later if the notebook runs quickly.
N_ITER_SEARCH = 2

# Cross-validation folds.## 5 originally
CV_SPLITS = 5

## 3. Load the data

This cell tries to load Excel, CSV, or Parquet files.

After loading, check:

- the number of rows and columns;
- whether the target column exists;
- whether the first few rows look sensible.

In [3]:
def load_tabular_data(path: Path) -> pd.DataFrame:
    """Load a tabular dataset from Excel, CSV, or Parquet."""
    suffix = path.suffix.lower()
    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file type: {suffix}")


df_raw = load_tabular_data(DATA_PATH)

print("Dataset shape:", df_raw.shape)
display(df_raw.head(2))

print("\nColumns:")
print(df_raw.columns.tolist())

Dataset shape: (301489, 22)


,s,ABP_Classification_Code,ABP_Classification_Desc,Communal Household Flag,(H) Age - Fine,(H) Household Composition,(H) Presence of Elderly Parent,(H) Number of Adults in Household,(H) Family Lifestage v3,(H) Length of Residency,...,(H) Mosaic UK 7 Type,(H) Mosaic UK 7 Type Label,(H) Affluence v2,(H) Water Poverty Flag,(H) Fuel Poverty v2 Flag,(H) Number of Children v3,(H) Household Income v3 - Bands,Station_Ground_Code,LSOA11CD,irs.Incident?
0,100120859623,RD04,Terraced,N,10,1.0,0,3,12.0,11,...,22,F22 Boomerang Boarders,13,0,N,0,2,JX32,E01028635,N
1,10002188486,RD02,Detached,N,11,1.0,0,4,12.0,11,...,10,C10 Wealthy Landowners,19,0,N,0,8,JX12,E01028762,N



Columns:
['s', 'ABP_Classification_Code', 'ABP_Classification_Desc', 'Communal Household Flag', '(H) Age - Fine', '(H) Household Composition', '(H) Presence of Elderly Parent', '(H) Number of Adults in Household', '(H) Family Lifestage v3', '(H) Length of Residency', '(H) Property Type 2011', '(H) Tenure 2011', '(H) Mosaic UK 7 Type', '(H) Mosaic UK 7 Type Label', '(H) Affluence v2', '(H) Water Poverty Flag', '(H) Fuel Poverty v2 Flag', '(H) Number of Children v3', '(H) Household Income v3 - Bands', 'Station_Ground_Code', 'LSOA11CD', 'irs.Incident?']


In [4]:
df_raw.rename(columns = {'s':'Addressbase UPRN','irs.Incident?':'Incident?'}, inplace =True)
df_raw.drop(columns = ['ABP_Classification_Code','(H) Mosaic UK 7 Type'], inplace = True)
df_raw.head(2)

,Addressbase UPRN,ABP_Classification_Desc,Communal Household Flag,(H) Age - Fine,(H) Household Composition,(H) Presence of Elderly Parent,(H) Number of Adults in Household,(H) Family Lifestage v3,(H) Length of Residency,(H) Property Type 2011,(H) Tenure 2011,(H) Mosaic UK 7 Type Label,(H) Affluence v2,(H) Water Poverty Flag,(H) Fuel Poverty v2 Flag,(H) Number of Children v3,(H) Household Income v3 - Bands,Station_Ground_Code,LSOA11CD,Incident?
0,100120859623,Terraced,N,10,1.0,0,3,12.0,11,4,0,F22 Boomerang Boarders,13,0,N,0,2,JX32,E01028635,N
1,10002188486,Detached,N,11,1.0,0,4,12.0,11,4,0,C10 Wealthy Landowners,19,0,N,0,8,JX12,E01028762,N


### added on 14/7/26
PLAN 2
Why not try to convert these numeric features directly into Boolean data types?



In [5]:
#df_raw.info()
df_raw[['(H) Fuel Poverty v2 Flag','(H) Presence of Elderly Parent','(H) Water Poverty Flag','Communal Household Flag']].info()

<class 'pandas.DataFrame'>
RangeIndex: 301489 entries, 0 to 301488
Data columns (total 4 columns):
 #   Column                          Non-Null Count   Dtype
---  ------                          --------------   -----
 0   (H) Fuel Poverty v2 Flag        301489 non-null  str  
 1   (H) Presence of Elderly Parent  301489 non-null  int64
 2   (H) Water Poverty Flag          301489 non-null  int64
 3   Communal Household Flag         301489 non-null  str  
dtypes: int64(2), str(2)
memory usage: 9.2 MB


In [6]:
df_raw[['(H) Fuel Poverty v2 Flag',
        '(H) Presence of Elderly Parent',
        '(H) Water Poverty Flag','Communal Household Flag']] = df_raw[['(H) Fuel Poverty v2 Flag','(H) Presence of Elderly Parent','(H) Water Poverty Flag','Communal Household Flag']].astype('bool')

In [7]:
df_raw[['(H) Fuel Poverty v2 Flag','(H) Presence of Elderly Parent','(H) Water Poverty Flag','Communal Household Flag']].info()

<class 'pandas.DataFrame'>
RangeIndex: 301489 entries, 0 to 301488
Data columns (total 4 columns):
 #   Column                          Non-Null Count   Dtype
---  ------                          --------------   -----
 0   (H) Fuel Poverty v2 Flag        301489 non-null  bool 
 1   (H) Presence of Elderly Parent  301489 non-null  bool 
 2   (H) Water Poverty Flag          301489 non-null  bool 
 3   Communal Household Flag         301489 non-null  bool 
dtypes: bool(4)
memory usage: 1.2 MB


### Reducing sample size to Station Ground Region as selected

In [8]:
#Station_Ground_Code='JX01'
#df_raw = df_raw[df_raw['Station_Ground_Code']==Station_Ground_Code]
#df_raw[['Station_Ground_Code','Incident?']].value_counts()

In [9]:

df_raw.shape
df_raw.head(4)
df_raw[['(H) Fuel Poverty v2 Flag','(H) Presence of Elderly Parent','(H) Water Poverty Flag','Communal Household Flag']].head(3)

,(H) Fuel Poverty v2 Flag,(H) Presence of Elderly Parent,(H) Water Poverty Flag,Communal Household Flag
0,True,False,False,True
1,True,False,False,True
2,True,False,False,True


## 4. Clean the target

The model needs the target to be numeric:

- `1` = incident / positive case;
- `0` = no incident / negative case.

Check the printed value counts carefully. If the mapping is wrong, fix it before continuing.

In [10]:
def clean_binary_target(series: pd.Series) -> pd.Series:
    """Convert a Yes/No or 0/1 target into clean integer 0/1 values."""
    # If already numeric 0/1, keep it simple.
    numeric = pd.to_numeric(series, errors="coerce")
    non_missing_numeric = numeric.dropna()
    if len(non_missing_numeric) > 0 and set(non_missing_numeric.unique()).issubset({0, 1}):
        return numeric.astype("Int64")

    # Otherwise use a conservative string mapping.
    normalised = series.astype(str).str.strip().str.lower()

    positive_values = {"yes", "y", "Y", "true", "1", "incident", "fire"}
    negative_values = {"no", "n", "N", "false", "0", "none", "no incident", "not incident"}

    mapped = normalised.map(lambda x: 1 if x in positive_values else 0 if x in negative_values else np.nan)
    return mapped.astype("Int64")


print("Original target values:")
display(df_raw[TARGET_RAW].value_counts(dropna=False))

# Create modelling dataframe.
df = df_raw.copy()

################################################################################################

df[TARGET] = clean_binary_target(df[TARGET_RAW])

# Drop rows where the target could not be mapped.
before = len(df)
df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = df[TARGET].astype(int)
after = len(df)

print(f"Rows before target cleaning: {before}")
print(f"Rows after target cleaning:  {after}")

print("\nCleaned target values:")
display(df[TARGET].value_counts(dropna=False).to_frame("count"))

Original target values:


Incident?
N    298648
Y      2841
Name: count, dtype: int64

Rows before target cleaning: 301489
Rows after target cleaning:  301489

Cleaned target values:


,count
target_incident,
0,298648
1,2841


## 5. Check class imbalance

This tells us how rare the positive class is.

If the positive class is rare, a model can have high accuracy while still being useless. That is why this notebook focuses on ranking metrics instead.

In [11]:

target_summary = df[TARGET].value_counts().sort_index().to_frame("count")
target_summary["proportion"] = target_summary["count"] / target_summary["count"].sum()
display(target_summary)

positive_rate = df[TARGET].mean()
print(f"Baseline positive rate: {positive_rate:.2%}")
print("This is the approximate success rate expected from random selection.")

,count,proportion
target_incident,,
0,298648,0.990577
1,2841,0.009423


Baseline positive rate: 0.94%
This is the approximate success rate expected from random selection.


## 6. Choose feature columns and avoid leakage

Some columns should not be used as model features.

Common examples:

- unique identifiers;
- post-incident information;
- columns that directly reveal the target;
- columns only known after the incident has happened;
- raw coordinate columns, unless you can justify their use.

The columns removed from modelling can still be kept later in the ranked output so the organisation can identify the records.

In [12]:
# TODO 3: Review this list. Add/remove columns based on your dataset.
# These columns will NOT be used as model features.
DROP_COLUMNS_AS_FEATURES = [
    TARGET_RAW,
    TARGET,
    "Addressbase UPRN",
 
    "Unnamed: 0",
    "Easting",
    "Northing",
    "FRSIncidentIdentifier",
    "IncidentCategory",
    "VictimsInvolved",
    "VictimType",
    "WasRescued",
     "LSOA11CD"
]

# These columns are useful for the final ranked output, if they exist.
ID_COLUMNS_FOR_OUTPUT = [
    "Addressbase UPRN",
    "Easting",
    "Northing",
    "LSOA11CD",
    "Property_Type",
    "Property_Description",
]

feature_columns = [
    col for col in df.columns
    if col not in DROP_COLUMNS_AS_FEATURES
]

id_columns = [
    col for col in ID_COLUMNS_FOR_OUTPUT
    if col in df.columns
]

print(f"Number of feature columns: {len(feature_columns)}")
print("\nFeature columns used by the model:")
print(feature_columns)

print("\nID/context columns kept for ranked output:")
print(id_columns)


Number of feature columns: 17

Feature columns used by the model:
['ABP_Classification_Desc', 'Communal Household Flag', '(H) Age - Fine', '(H) Household Composition', '(H) Presence of Elderly Parent', '(H) Number of Adults in Household', '(H) Family Lifestage v3', '(H) Length of Residency', '(H) Property Type 2011', '(H) Tenure 2011', '(H) Mosaic UK 7 Type Label', '(H) Affluence v2', '(H) Water Poverty Flag', '(H) Fuel Poverty v2 Flag', '(H) Number of Children v3', '(H) Household Income v3 - Bands', 'Station_Ground_Code']

ID/context columns kept for ranked output:
['Addressbase UPRN', 'LSOA11CD']


## 7. Split the data

We use three sets:

- **Train**: fit the model and tune hyperparameters.
- **Validation**: compare the tuned model and inspect ranking performance.
- **Test**: final honest evaluation, used only once at the end.

Because the target is imbalanced, we use stratified splits so each set has a similar positive rate.

In [13]:
X = df[feature_columns].copy()
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 301489 entries, 0 to 301488
Data columns (total 17 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   ABP_Classification_Desc            301489 non-null  str    
 1   Communal Household Flag            301489 non-null  bool   
 2   (H) Age - Fine                     301489 non-null  int64  
 3   (H) Household Composition          299873 non-null  float64
 4   (H) Presence of Elderly Parent     301489 non-null  bool   
 5   (H) Number of Adults in Household  301489 non-null  int64  
 6   (H) Family Lifestage v3            299873 non-null  float64
 7   (H) Length of Residency            301489 non-null  int64  
 8   (H) Property Type 2011             301489 non-null  int64  
 9   (H) Tenure 2011                    301489 non-null  int64  
 10  (H) Mosaic UK 7 Type Label         301489 non-null  str    
 11  (H) Affluence v2                   301489 non-null

In [14]:
#X = df[feature_columns].astype(str)
#X.info()

In [15]:
# X = df[feature_columns].copy() --> suppressed since I am trying out the code: 
### X = df[feature_columns].astype(str) in previous cell
y = df[TARGET].copy()
ids = df[id_columns].copy() if id_columns else pd.DataFrame(index=df.index)

# First split: train+validation vs test.
X_train_val, X_test, y_train_val, y_test, ids_train_val, ids_test = train_test_split(
    X,
    y,
    ids,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

# Second split: train vs validation.
X_train, X_valid, y_train, y_valid, ids_train, ids_valid = train_test_split(
    X_train_val,
    y_train_val,
    ids_train_val,
    test_size=VALIDATION_SIZE_WITHIN_TRAIN,
    stratify=y_train_val,
    random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame({
    "rows": [len(y_train), len(y_valid), len(y_test)],
    "positives": [int(y_train.sum()), int(y_valid.sum()), int(y_test.sum())],
    "positive_rate": [y_train.mean(), y_valid.mean(), y_test.mean()],
}, index=["train", "validation", "test"])

display(split_summary)

,rows,positives,positive_rate
train,180893,1705,0.009425
validation,60298,568,0.009420
test,60298,568,0.009420


## 8. Build the preprocessing pipeline

The preprocessing step handles:

- missing numeric values;
- missing categorical values;
- scaling numeric features;
- one-hot encoding categorical features.

Putting preprocessing inside the pipeline helps avoid data leakage during cross-validation.

In [16]:
numeric_features = X_train.select_dtypes(include=["int", "number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "string"]).columns.tolist()

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Compatible with different scikit-learn versions.
try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=True)

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", onehot),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

Numeric features: 14
Categorical features: 3


In [17]:
X_train[categorical_features].info()

<class 'pandas.DataFrame'>
Index: 180893 entries, 151069 to 19372
Data columns (total 3 columns):
 #   Column                      Non-Null Count   Dtype
---  ------                      --------------   -----
 0   ABP_Classification_Desc     180893 non-null  str  
 1   (H) Mosaic UK 7 Type Label  180893 non-null  str  
 2   Station_Ground_Code         180893 non-null  str  
dtypes: str(3)
memory usage: 5.5 MB


In [18]:
X_train[numeric_features].info()

<class 'pandas.DataFrame'>
Index: 180893 entries, 151069 to 19372
Data columns (total 14 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Communal Household Flag            180893 non-null  bool   
 1   (H) Age - Fine                     180893 non-null  int64  
 2   (H) Household Composition          179923 non-null  float64
 3   (H) Presence of Elderly Parent     180893 non-null  bool   
 4   (H) Number of Adults in Household  180893 non-null  int64  
 5   (H) Family Lifestage v3            179923 non-null  float64
 6   (H) Length of Residency            180893 non-null  int64  
 7   (H) Property Type 2011             180893 non-null  int64  
 8   (H) Tenure 2011                    180893 non-null  int64  
 9   (H) Affluence v2                   180893 non-null  int64  
 10  (H) Water Poverty Flag             180893 non-null  bool   
 11  (H) Fuel Poverty v2 Flag           180893 non-null 

## 9. Helper functions for ranking evaluation

These functions are provided for you.

The most important table is the **top-K table**. It answers:

> If we inspect only the top K highest-scored records, how many true positives do we capture?

Important distinction:

- `precision@K` = true positives in top K / K.
- `recall@K` = true positives in top K / all positives in the dataset.

So top-50 recall can look low even when precision is high, because the denominator is all positives, not 50.

In [19]:
def get_scores(model, X_data):
    """Return the model's positive-class probability scores."""
    return model.predict_proba(X_data)[:, 1]


def top_k_capture_table(y_true, scores, k_values, label="data"):
    """Create a top-K ranking evaluation table."""
    y_array = np.asarray(y_true).astype(int)
    scores_array = np.asarray(scores)

    n_records = len(y_array)
    total_positives = int(y_array.sum())
    baseline_rate = y_array.mean()

    order = np.argsort(scores_array)[::-1]
    y_sorted = y_array[order]

    rows = []
    for k in k_values:
        k_eff = min(k, n_records)
        positives_in_top_k = int(y_sorted[:k_eff].sum())

        precision_at_k = positives_in_top_k / k_eff if k_eff > 0 else np.nan
        recall_at_k = positives_in_top_k / total_positives if total_positives > 0 else np.nan
        lift_at_k = precision_at_k / baseline_rate if baseline_rate > 0 else np.nan

        # Top K cannot capture more than K positives.
        max_possible_recall = min(k_eff, total_positives) / total_positives if total_positives > 0 else np.nan
        pct_of_max_possible = recall_at_k / max_possible_recall if max_possible_recall > 0 else np.nan

        rows.append({
            "dataset": label,
            "k": k_eff,
            "positives_in_top_k": positives_in_top_k,
            "precision_at_k": precision_at_k,
            "recall_at_k": recall_at_k,
            "max_possible_recall_at_k": max_possible_recall,
            "pct_of_max_possible_recall": pct_of_max_possible,
            "lift_at_k": lift_at_k,
        })

    return pd.DataFrame(rows)


def cumulative_gain_frame(y_true, scores):
    """Return data for a cumulative gains curve."""
    y_array = np.asarray(y_true).astype(int)
    scores_array = np.asarray(scores)

    order = np.argsort(scores_array)[::-1]
    y_sorted = y_array[order]

    cumulative_positives = np.cumsum(y_sorted)
    total_positives = y_sorted.sum()

    return pd.DataFrame({
        "inspected_fraction": np.arange(1, len(y_sorted) + 1) / len(y_sorted),
        "cumulative_capture_rate": cumulative_positives / total_positives if total_positives > 0 else np.nan,
    })

## 10. Fit a simple baseline Random Forest

This gives a starting point before hyperparameter tuning.

Do not worry if performance is not perfect. We are checking whether the model has useful ranking signal.

In [20]:
#### The problem being that the algorithm does not like mixed data types such as str and object being in the same list.
#### On 23/6/26 this was fixed by 
string_cols = X_train.select_dtypes(include=["string","object"]).columns
X_train[string_cols] = X_train[string_cols].astype("string")


In [21]:
baseline_rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

## FIT
baseline_rf.fit(X_train, y_train)

baseline_valid_scores = get_scores(baseline_rf, X_valid)

print("Baseline Random Forest — validation")
print(f"ROC-AUC:           {roc_auc_score(y_valid, baseline_valid_scores):.3f}")
print(f"Average Precision: {average_precision_score(y_valid, baseline_valid_scores):.3f}")

display(top_k_capture_table(y_valid, baseline_valid_scores, TOP_K_VALUES, label="validation"))


Baseline Random Forest — validation
ROC-AUC:           0.776
Average Precision: 0.278


,dataset,k,positives_in_top_k,precision_at_k,recall_at_k,max_possible_recall_at_k,pct_of_max_possible_recall,lift_at_k
0,validation,50,49,0.980,0.086268,0.088028,0.980,104.035282
1,validation,100,65,0.650,0.114437,0.176056,0.650,69.002993
2,validation,200,115,0.575,0.202465,0.352113,0.575,61.041109
3,validation,500,200,0.400,0.352113,0.880282,0.400,42.463380


In [22]:
baseline_rf

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](17,)","['ABP_Classification_Desc','Communal Household Flag','(H) Age - Fine',..., '(H) Number of Children v3','(H) Household Income v3 - Bands', 'Station_Ground_Code']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,17
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcol

## 11. Tune the Random Forest with RandomizedSearchCV

This searches a wider set of Random Forest settings than a small manual grid.

We tune using **Average Precision** because the target is imbalanced and we care about ranking positives near the top. We still report ROC-AUC because it tells us whether the model has general ranking signal.

Start with `N_ITER_SEARCH = 25`. If the notebook runs quickly, increase it to 50 or 75.

In [23]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])
'''
# first run
param_distributions = {
    "model__n_estimators": randint(300, 1200),
    "model__max_depth": [None, 5, 8, 12, 16, 24, 32],
    "model__min_samples_split": randint(2, 50),
    "model__min_samples_leaf": randint(1, 25),
    "model__max_features": ["sqrt", "log2", 0.3, 0.5, 0.7],
    "model__class_weight": [None, "balanced", "balanced_subsample"],
    "model__bootstrap": [True, False],

}
-- Output
--> Best CV Average Precision: 0.22204475004176244
-- > Best parameters:
  model__bootstrap: False
  model__class_weight: balanced_subsample
  model__max_depth: 32
  model__max_features: 0.3
  model__min_samples_leaf: 11
  model__min_samples_split: 25
  model__n_estimators: 672

'''

# second run:
param_distributions = {
    "model__n_estimators": randint(500, 1000),
    "model__max_depth": [24, 32, 36, 38, 40],
    "model__min_samples_split": randint(15, 35),
    "model__min_samples_leaf": randint(5, 15),
    "model__max_features": [0.2, 0.3, 0.4, 0.5,0.6],
    "model__class_weight": ["balanced_subsample"],
    "model__bootstrap": [ False],
}



minority_count = int(y_train.value_counts().min())
n_splits = max(2, min(CV_SPLITS, minority_count))

cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=RANDOM_STATE,
)

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_distributions,
    n_iter=N_ITER_SEARCH,
    scoring={
        "roc_auc": "roc_auc",
        "average_precision": "average_precision",
    },
    refit="average_precision",
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
    return_train_score=True,
)


In [24]:
import time
start_time = time.time()


In [ ]:
rf_search.fit(X_train, y_train)


Fitting 5 folds for each of 2 candidates, totalling 10 fits


In [ ]:
print("Best CV Average Precision:", rf_search.best_score_)
print("Best parameters:")
for key, value in rf_search.best_params_.items():
    print(f"  {key}: {value}")


In [ ]:
print("pipeline_Random_Forest fitted\n--- %s seconds ---" % (time.time() - start_time))

## 12. Evaluate the tuned Random Forest on validation data

This checks whether tuning improved the model on unseen validation data.

Focus especially on:

- ROC-AUC;
- Average Precision;
- positives captured in the top 50, 100, and 200;
- lift compared with random selection;
- percentage of the maximum possible recall@K.

In [ ]:
best_rf = rf_search.best_estimator_
tuned_valid_scores = get_scores(best_rf, X_valid)

print("Tuned Random Forest — validation")
print(f"ROC-AUC:           {roc_auc_score(y_valid, tuned_valid_scores):.3f}")
print(f"Average Precision: {average_precision_score(y_valid, tuned_valid_scores):.3f}")

valid_top_k = top_k_capture_table(y_valid, tuned_valid_scores, TOP_K_VALUES, label="validation")
display(valid_top_k)

## 13. Compare baseline and tuned model

Use this section to decide whether tuning actually helped.

A model is not automatically better just because it was tuned. It should improve the validation results or provide a better operational ranking.

In [ ]:
comparison_rows = []

for model_name, scores in [
    ("Baseline RF", baseline_valid_scores),
    ("Tuned RF", tuned_valid_scores),
]:
    top_k = top_k_capture_table(y_valid, scores, TOP_K_VALUES, label="validation")
    top_50_row = top_k[top_k["k"] == min(50, len(y_valid))].iloc[0]

    comparison_rows.append({
        "model": model_name,
        "roc_auc": roc_auc_score(y_valid, scores),
        "average_precision": average_precision_score(y_valid, scores),
        "top_50_true_positives": top_50_row["positives_in_top_k"],
        "precision_at_50": top_50_row["precision_at_k"],
        "recall_at_50": top_50_row["recall_at_k"],
        "pct_of_max_possible_recall_at_50": top_50_row["pct_of_max_possible_recall"],
        "lift_at_50": top_50_row["lift_at_k"],
    })

comparison = pd.DataFrame(comparison_rows)
display(comparison)

## 14. Plot the validation cumulative gains curve

This plot shows how quickly the model captures true positives as more records are inspected.

A useful model should rise faster than the diagonal/random line.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


baseline_gain_valid = cumulative_gain_frame(y_valid, baseline_valid_scores)
tuned_gain_valid = cumulative_gain_frame(y_valid, tuned_valid_scores)
'''
ax1 = baseline_gain_valid.plot(
    x="inspected_fraction",
    y="cumulative_capture_rate",
    legend=False,
    title="Cumulative gains curve — validation — Baseline Random Forest",
)

ax2 = tuned_gain_valid.plot(
    x="inspected_fraction",
    y="cumulative_capture_rate",
    legend=False,
    title="Cumulative gains curve — validation — Tuned Random Forest",
)
'''


plt.plot(baseline_gain_valid, label = 'BaseLine_RF', color = 'blue')
plt.plot(tuned_gain_valid,    label = "Tuned_RF", color = 'red')


plt.legend(loc="lower right")

plt.xlabel("Fraction of records inspected")
plt.ylabel("Fraction of positives captured")
plt.title('Cumulative gains curve — validation')
plt.show()

In [ ]:
'''
baseline_gain_valid = cumulative_gain_frame(y_valid, baseline_valid_scores)
tuned_gain_valid = cumulative_gain_frame(y_valid, tuned_valid_scores)

ax = baseline_gain_valid.plot(
    x="inspected_fraction",
    y="cumulative_capture_rate",
    legend=False,
    title="Cumulative gains curve — validation — Baseline Random Forest",
)

ax = tuned_gain_valid.plot(
    x="inspected_fraction",
    y="cumulative_capture_rate",
    legend=False,
    title="Cumulative gains curve — validation — Tuned Random Forest",
)


ax.set_xlabel("Fraction of records inspected")
ax.set_ylabel("Fraction of positives captured")
plt.show()
'''

In [ ]:
'''
gain_valid = cumulative_gain_frame(y_valid, tuned_valid_scores)

ax = gain_valid.plot(
    x="inspected_fraction",
    y="cumulative_capture_rate",
    legend=False,
    title="Cumulative gains curve — validation — tuned Random Forest",
)
ax.set_xlabel("Fraction of records inspected")
ax.set_ylabel("Fraction of positives captured")
plt.show()
'''

## 15. Final test evaluation

Only run this after you have finished choosing the model using the validation set.

Do not tune the model again after looking at the test results.

In [ ]:
final_model = best_rf

test_scores = get_scores(final_model, X_test)

print("Final tuned Random Forest — test")
print(f"ROC-AUC:           {roc_auc_score(y_test, test_scores):.3f}")
print(f"Average Precision: {average_precision_score(y_test, test_scores):.3f}")

test_top_k = top_k_capture_table(y_test, test_scores, TOP_K_VALUES, label="test")
display(test_top_k)

## 16. Create a ranked test output

This creates a table sorted from highest predicted risk to lowest predicted risk.

The top rows are the records the organisation would prioritise for inspection or intervention.

In [ ]:
ids_test

In [ ]:
ranked_test = ids_test.copy()
ranked_test["true_target"] = y_test.values
ranked_test["risk_score"] = test_scores
ranked_test["risk_rank"] = ranked_test["risk_score"].rank(method="first", ascending=False).astype(int)

ranked_test = ranked_test.sort_values("risk_score", ascending=False)

display(ranked_test)#.head(50))

# Save outputs for reporting.
ranked_test.to_csv("ranked_test_output.csv", index=False)
#ranked_test.head(50).to_csv("top_50_ranked_test_output.csv", index=False)

#print("Saved ranked_test_output.csv")
#print("Saved top_50_ranked_test_output.csv")

## 17. How to explain low recall@50

Use this explanation if your top-50 recall looks low.

Recall@50 is calculated as:

```text
true positives in top 50 / all true positives in the dataset
```

So if the top 50 contains 38 true positives, precision is high:

```text
precision@50 = 38 / 50 = 0.76
```

But recall may still look low if there are many positives overall. For example, if there are 200 positives:

```text
recall@50 = 38 / 200 = 0.19
```

That does not mean the model is useless. It means top 50 is too small to capture most positives.

That is why this notebook also reports:

```text
pct_of_max_possible_recall
```

This shows how close the model is to the best possible result for that value of K.

## 18. Interpretation prompts

Answer these in markdown after running the notebook.

1. What is the positive class and why is the problem imbalanced?
2. Why is accuracy not enough for this task?
3. What was the validation ROC-AUC?
4. What was the validation Average Precision?
5. In the validation top 50, how many true positives were captured?
6. What was precision@50?
7. What was recall@50?
8. What percentage of the maximum possible recall@50 did the model achieve?
9. Did tuning improve the baseline Random Forest?
10. Based on the test results, should the organisation use top 50, top 100, or top 200?
11. What extra data might improve the ranking?
12. What are the limitations of using this model operationally?

Suggested conclusion structure:

> The model should be used as a prioritisation tool, not as an automatic decision-maker. The most useful metric is whether the top-ranked records contain more true incidents than random selection. The final top-K results suggest that [...]. However, the model is limited by [...], so future work should [...].

In [ ]:
### Extra part.
#### Join geolocation result:
Gazetter = pd.read_excel('C:/Users/pw347789/OneDrive - Oxfordshire County Council/Desktop/Cambridge Spark/Documents for June 2026/OXFORDSHIRE_GAZETTER.xlsx')

In [ ]:
Gazetter = pd.DataFrame(Gazetter)
Gazetter.head(2)

In [ ]:
#Gazetter.drop(columns = ['BUILDINGNAME' , 'BUILDINGNUMBER'])
                         
                        #, 'STREETNAME1','STREETNAME2', 'AREANAME1', 'AREANAME2', 'POSTCODE', 'POSTTOWN',
 #      'POSTCOUNTY'], inplace = True)

In [ ]:
ranked_geolocs = Gazetter.merge(ranked_test, left_on = 'FCL_URN', right_on = 'Addressbase UPRN', how = 'right')

In [ ]:
ranked_geolocs.drop(columns = ['BUILDINGNAME' , 'BUILDINGNUMBER'], inplace = True)
ranked_geolocs           

In [ ]:

Station_Ground_Code

In [ ]:
ranked_geolocs.to_csv(f'H:/OXFS Share/AdamMason/PROJECTS/C_SPark_Home_Safety/Ranked_Property_STATIONS/ranked_geolocs_for_map_{Station_Ground_Code}.csv')

# 16. Testing:

In [ ]:
from sklearn.metrics import precision_score, roc_auc_score, mean_squared_error, accuracy_score
from sklearn.metrics import recall_score, classification_report, roc_curve, confusion_matrix
from sklearn import metrics

### TRAINING DATA

In [ ]:
# Model Performance
# we can get performance of the model on the TRAIN data set from 
y_train_pred_best_rf =  best_rf.predict(X_train)
y_train_pred_best_rf

In [ ]:
## 5. Evaluate on TRAINING SET: 
from sklearn.metrics import classification_report
target_names = ['0', '1']
print("best_rf(X_train)\n\nClassification Report:")
print(classification_report(y_train, y_train_pred_best_rf, target_names=target_names))

## TEST DATA

In [ ]:
# 5. Evaluate on TEST SET: 
y_test_pred_best_rf = best_rf.predict(X_test)
y_test_pred_best_rf

target_names = ['0', '1']
print("best_rf(X_test)\n\nClassification Report:")
print(classification_report(y_test, y_test_pred_best_rf, target_names=target_names))

## Accuracy Scores:

In [ ]:
best_rf_pred_RF_ACCURACY =  metrics.accuracy_score( y_train,  y_train_pred_best_rf)
best_rf_pred_RF_PRECISION = metrics.precision_score(y_train,  y_train_pred_best_rf)
best_rf_pred_RF_RECALL =    metrics.recall_score(   y_train,  y_train_pred_best_rf)
best_rf_pred_RF_PF1 =       metrics.f1_score(       y_train,  y_train_pred_best_rf)

In [ ]:
d = {"accuracy_score":  [metrics.accuracy_score(y_train, y_train_pred_best_rf),  metrics.accuracy_score( y_test, y_test_pred_best_rf)], 
     "precision_score": [metrics.precision_score(y_train, y_train_pred_best_rf), metrics.precision_score(y_test, y_test_pred_best_rf)],
     "recall_score":    [metrics.recall_score(y_train, y_train_pred_best_rf),    metrics.recall_score(   y_test, y_test_pred_best_rf)],
     "f1_score":        [metrics.f1_score(y_train, y_train_pred_best_rf),        metrics.f1_score(       y_test, y_test_pred_best_rf)]    
    }
print("RANDOM FOREST baseline: With respect to target variable = 1\n")
RF_metrics = pd.DataFrame(data=d, index=['TRAIN', 'TEST'])
RF_metrics = RF_metrics.T
RF_metrics


In [ ]:
plt.figure(figsize = (10,5))
#RF_metrics[['accuracy_score','precision_score']].plot(kind = 'bar', legend = True)
#RF_metrics[['recall_score',	'f1_score']].plot(kind = 'bar', legend = True)

RF_metrics[['TRAIN','TEST']].plot(kind = 'bar', legend = True)
plt.title("PERFRMANCE METRICS\nRANDOM FOREST baseline model:\nTRAIN : TEST") 
#plt.legend(loc = " right")
plt.legend(loc=(0.8,1.1))
plt.xlabel('Data set')
plt.ylabel('Measure [0,1]')
plt.show()

## ROC metrics

TEST data Prediction Probabilities:

## BASELINE

In [ ]:
y_test_pred_proba_baseline_rf =  baseline_rf.predict_proba(X_test)[:,1]
y_test_pred_proba_baseline_rf

In [ ]:
logit_roc_auc_baseline        = roc_auc_score(y_test,    y_test_pred_proba_baseline_rf)
print(f'ROC-AUC_score[y_test]: {logit_roc_auc_baseline}')

In [ ]:
fpr_baseline_rf, tpr_baseline_rf, thresholds = roc_curve(y_test,    y_test_pred_proba_baseline_rf)

In [ ]:
plt.figure()
plt.plot(fpr_baseline_rf, tpr_baseline_rf, label='Random Forest (area = %0.2f)' % logit_roc_auc_baseline )
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve:\nRANDOM FOREST baseline model\ny_test_pred_proba_baseline_rf')
plt.legend(loc="lower right")
plt.show()

In [ ]:
x = RF_metrics.T['precision_score'][1:][0:] 
#y = RF_metrics['precision_score'][0:][0:]



## TUNED ROC:  
#### logit_roc_auc_best_rf

In [ ]:
y_test_pred_proba_best_rf =  best_rf.predict_proba(X_test)[:,1]
y_test_pred_proba_best_rf

In [ ]:
logit_roc_auc_best_rf       = roc_auc_score(y_test,    y_test_pred_proba_best_rf)
print(f'ROC-AUC_score[y_test]: {logit_roc_auc_best_rf}')

In [ ]:
fpr_best_rf, tpr_best_rf, thresholds = roc_curve(y_test,    y_test_pred_proba_best_rf)

In [ ]:
plt.figure()
plt.plot(fpr_best_rf, tpr_best_rf, label='Random Forest (area = %0.2f)' % logit_roc_auc_best_rf)
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve:\nRANDOM FOREST: TUNED model model\ny_test_pred_proba_best_rf')
plt.legend(loc="lower right")
plt.show()

In [ ]:

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve:\nRANDOM FOREST \nbaseline model : best_rf model')


plt.plot(fpr_baseline_rf, tpr_baseline_rf, label='Random Forest (area = %0.2f)' % logit_roc_auc_baseline, color = 'blue')
plt.plot(fpr_best_rf, tpr_best_rf, label='Random Forest (area = %0.2f)' % logit_roc_auc_best_rf, color = 'red')
plt.plot([0, 1], [0, 1],'r--')

plt.legend(labels = ['baseline_rf','tuned_rf'], loc="lower right")
plt.show()

In [ ]:
data = {
  "Baseline_ROC_Score": logit_roc_auc_baseline,
  "Tuned_RF_ROC_Score": logit_roc_auc_best_rf
}
data

